# Kaggle GPU runtime -- CathAction unfreeze-backbone smoke test

Upload this notebook to Kaggle (New Notebook > File > Upload Notebook), then in the right sidebar:
- **Settings > Accelerator**: GPU T4 x2 (or any GPU)
- **Add-ons > Data**: attach dataset `sofiazowormazabal/cathaction`
- **Add-ons > Secrets**: add `WANDB_API_KEY` if you want online logging (skip to run offline)

Push your local changes first -- the clone step below pulls from GitHub, it does not see uncommitted local edits.

In [ ]:
!nvidia-smi

In [ ]:
!test -d /kaggle/working/Phillips_UC2/.git && git -C /kaggle/working/Phillips_UC2 pull || git clone https://github.com/sormazabal/Phillips_UC2.git /kaggle/working/Phillips_UC2
%cd /kaggle/working/Phillips_UC2
!pip install -q -r requirements.txt

## Dataset

Kaggle's dataset mount path has varied (sometimes `/kaggle/input/cathaction`, sometimes `/kaggle/input/datasets/<owner>/cathaction`) -- find it explicitly rather than guessing, then pass it to `train.py` via `--data_root`.

In [ ]:
!find /kaggle/input -maxdepth 6 -iname "train.json"

In [ ]:
# Set this to the directory printed above, minus the trailing "/coco/annotations/train.json"
%env DATA_ROOT=/kaggle/input/datasets/sofiazowormazabal/cathaction

In [ ]:
# Optional: WandB key from Kaggle Secrets (Add-ons > Secrets). Falls back to offline if not set,
# same as the Colab notebook's .env fallback -- offline never blocks on an interactive login prompt.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    os.environ["WANDB_MODE"] = "offline"

## Smoke test

1 epoch, 20 train batches, 5 val batches -- confirms the unfrozen mit-b2 backbone fits in VRAM and trains without crashing before committing to the full 40-epoch run. Check the printed model summary for **`Trainable params` ~= 24-26M** (not ~2M), and that it completes without an OOM traceback.

In [ ]:
%env PYTHONPATH=/kaggle/working/Phillips_UC2
!python scripts/train.py --config config_cathaction.yaml --data_root $DATA_ROOT --max_epochs 1 --limit_train_batches 20 --limit_val_batches 5 --checkpoint_dir /kaggle/working/CathAction_checkpoints

## Full run (only after the smoke test passes)

Baseline to beat: **val dice 0.4214** (peak-threshold dice measured on the old frozen-backbone checkpoint). Kaggle sessions cap at 12h/9h depending on accelerator -- if a 40-epoch run risks running long, lower `max_epochs` per session and `--resume` from `checkpoints/last.ckpt`, or download checkpoints from `/kaggle/working/CathAction_checkpoints` via the notebook's Output tab between sessions.

In [ ]:
%env PYTHONPATH=/kaggle/working/Phillips_UC2
!python scripts/train.py --config config_cathaction.yaml --data_root $DATA_ROOT --checkpoint_dir /kaggle/working/CathAction_checkpoints